# Parte 3 - Cruce por ubigeo y análisis
**Pregunta:** ¿El Estado declara emergencia donde más llueve? (Temporada 2024)

In [45]:
import pandas as pd

### Paso 1 - Crear datos/ubigeos.csv
Usamos los ubigeos departamentales oficiales del INEI.
Escribimos el ubigeo como **texto** ("01", no 1) para no perder el cero.

In [46]:
#Ubigeos oficiales del INEI.
ubigeos = pd.DataFrame({"ubigeo":["01","02","03","04","05","06","07","08","09","10","11","12","13","14","15","16","17","18","19","20","21","22","23","24","25"], "departamento": ["Amazonas","Áncash","Apurímac","Arequipa","Ayacucho","Cajamarca","Callao","Cusco","Huancavelica","Huánuco","Ica","Junín","La Libertad","Lambayeque","Lima","Loreto","Madre de Dios","Moquegua","Pasco","Piura","Puno","San Martín","Tacna","Tumbes","Ucayali"]})
#index=False: no guardar el índice; utf-8-sig: tildes correctas en excel
ubigeos.to_csv("datos/ubigeos.csv", index=False, encoding="utf-8-sig")
print("filas:", len(ubigeos)) #control: deben ser 25 departamentos
ubigeos

filas: 25


,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho
5,06,Cajamarca
6,07,Callao
7,08,Cusco
8,09,Huancavelica
9,10,Huánuco


### Paso 2 - Leer "ubigeos.csv" dos veces
primero con pd.read_csv normal y luego indicando "dtype={"ubigeo":str}

In [47]:
ubigeos_normal=pd.read_csv("datos/ubigeos.csv")
print(ubigeos_normal.dtypes)
ubigeos_normal.head(10)

ubigeo          int64
departamento      str
dtype: object


,ubigeo,departamento
0,1,Amazonas
1,2,Áncash
2,3,Apurímac
3,4,Arequipa
4,5,Ayacucho
5,6,Cajamarca
6,7,Callao
7,8,Cusco
8,9,Huancavelica
9,10,Huánuco


In [48]:
ubigeos_texto=pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})
print(ubigeos_texto.dtypes)
ubigeos_texto.head(10)

ubigeo          str
departamento    str
dtype: object


,ubigeo,departamento
0,01,Amazonas
1,02,Áncash
2,03,Apurímac
3,04,Arequipa
4,05,Ayacucho
5,06,Cajamarca
6,07,Callao
7,08,Cusco
8,09,Huancavelica
9,10,Huánuco


In [49]:
#comparamos ambas lecturas en una sola tabla
comparacion=pd.DataFrame({"departamento": ubigeos_texto["departamento"],"ubigeo_normal":ubigeos_normal["ubigeo"],"ubigeo_texto":ubigeos_texto["ubigeo"]})
print(comparacion.head(10))
#prueba concreta: buscar Amazonas por su ubigeo "01"
print("\n¿Encuentra '01' leyendo normal?:", (ubigeos_normal["ubigeo"] == "01").sum())
print("¿Encuentra '01' leyendo como texto?:", (ubigeos_texto["ubigeo"] == "01").sum())

   departamento  ubigeo_normal ubigeo_texto
0      Amazonas              1           01
1        Áncash              2           02
2      Apurímac              3           03
3      Arequipa              4           04
4      Ayacucho              5           05
5     Cajamarca              6           06
6        Callao              7           07
7         Cusco              8           08
8  Huancavelica              9           09
9       Huánuco             10           10

¿Encuentra '01' leyendo normal?: 0
¿Encuentra '01' leyendo como texto?: 1


##### ¿Por qué es importante leer el ubigeo como texto?
- **Qué pasó:** al leer el archivo de forma normal, pandas convirtió el ubigeo de Amazonas "01" en 1. Los departamentos del 01 al 09 perdieron el cero de la izquierda.
- **Qué es el ubigeo:** es un código de identificación, debe guardarse como texto.
- **La prueba:** al buscar "01", la lectura normal encontró 0 coincidencias, mientras que la lectura como texto encontró 1 (Amazonas). Si cruzamos tablas con el ubigeo como número, Amazonas se perdería.
- **Formato oficial:** el INEI usa siempre dos dígitos "01", así debe mantenerse en la tabla_final.csv
- **Decisión:** leer siempre con: `dtype={"ubigeo": str}`

### Paso 3 - Agregar el ubigeo a cada tabla
Agregamos el `ubigeo` buscando el nombre del departamento en `ubigeos.csv`.
Usamos `how=left` para no perder ningún departamento: si alguno no encuentra su ubigeo, queda vacío (NaN) y lo detectamos en el paso 4.

### 3a. Decretos por departamento

In [50]:
#Leemos los ubigeos SIEMPRE como texto (decisión del paso 2)
ubigeos = pd.read_csv("datos/ubigeos.csv", dtype={"ubigeo": str})
#Tabla de la Parte 1: declaratorias y prórrogas por departamento
decretos=pd.read_csv("datos/decretos_por_departamento.csv")
print("Decretos:", decretos.shape) #control: 25 filas, 3 columnas
#Agregamos el ubigeo buscando por nombre: how="left" para que no se pierdan los departamentos que no tengan decretos
decretos_ub = decretos.merge(ubigeos, how="left", on="departamento")
#Ordenamos las columnas para que ubigeo quede primero
decretos_ub = decretos_ub[["ubigeo", "departamento", "declaratorias", "prorrogas"]]
decretos_ub.head()

Decretos: (25, 3)


,ubigeo,departamento,declaratorias,prorrogas
0,01,Amazonas,4,2
1,02,Áncash,5,5
2,03,Apurímac,7,4
3,04,Arequipa,7,6
4,05,Ayacucho,7,5


### 3b. Lluvias por departamento

In [51]:
# Tabla de la Parte 2: lluvia de la temporada en la capital de cada departamento
lluvias = pd.read_csv("datos/lluvias_por_departamento.csv")
print("Lluvias:", lluvias.shape) # control: 25 filas, 6 columnas
# Agregamos el ubigeo buscando por nombre, igual que con decretos
lluvias_ub = lluvias.merge(ubigeos, how="left", on="departamento")
lluvias_ub = lluvias_ub[["ubigeo", "departamento", "capital", "latitud", "longitud",
                         "lluvia_total_mm", "dias_lluvia_fuerte"]]
lluvias_ub.head()

Lluvias: (25, 6)


,ubigeo,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,01,Amazonas,Chachapoyas,-6.23169,-77.86903,455.0,0
1,02,Áncash,Huaraz,-9.52614,-77.52869,1058.3,3
2,03,Apurímac,Abancay,-13.63426,-72.88380,670.6,2
3,04,Arequipa,Arequipa,-16.39899,-71.53747,235.3,1
4,05,Ayacucho,Ayacucho,-13.16380,-74.22345,467.5,1


### Paso 4 - Verificación: ¿algún departamento no encontró su ubigeo?
### 4a. Decretos

In [52]:
# Si un nombre está escrito distinto (sin tilde, por ejemplo), su ubigeo queda vacío (NaN)
sin_ubigeo = decretos_ub[decretos_ub["ubigeo"].isna()]
print("Departamentos de decretos sin ubigeo:", len(sin_ubigeo))
print(sin_ubigeo)
# Revisión al revés: ¿algún ubigeo oficial no aparece en decretos?
faltan = set(ubigeos["departamento"]) - set(decretos["departamento"])
print("Departamentos oficiales que no están en decretos:", faltan)
# Control final
print("Filas:", len(decretos_ub), "| ubigeos vacíos:", decretos_ub["ubigeo"].isna().sum())

Departamentos de decretos sin ubigeo: 0
Empty DataFrame
Columns: [ubigeo, departamento, declaratorias, prorrogas]
Index: []
Departamentos oficiales que no están en decretos: set()
Filas: 25 | ubigeos vacíos: 0


**Resultado 4a:** los 25 departamentos de `decretos_por_departamento.csv` encontraron su ubigeo.

### 4b. Verificación de lluvias

In [53]:
sin_ubigeo_ll = lluvias_ub[lluvias_ub["ubigeo"].isna()]
print("Departamentos de lluvias sin ubigeo:", len(sin_ubigeo_ll))
print(sin_ubigeo_ll)
print("Oficiales que no están en lluvias:", set(ubigeos["departamento"]) - set(lluvias["departamento"]))
print("Filas:", len(lluvias_ub), "| ubigeos vacíos:", lluvias_ub["ubigeo"].isna().sum())

Departamentos de lluvias sin ubigeo: 0
Empty DataFrame
Columns: [ubigeo, departamento, capital, latitud, longitud, lluvia_total_mm, dias_lluvia_fuerte]
Index: []
Oficiales que no están en lluvias: set()
Filas: 25 | ubigeos vacíos: 0


Resultado 4b: los 25 departamentos de la tabla de lluvias encontraron su ubigeo. No se corrigió nombres ya que en la Parte 2 se usa la misma lista oficial con tildes. El Callao, fue agregado en la Parte 2 y encontró su ubigeo (07).

### Paso 5 - Cruce por ubigeo
Partimos de la lista oficial de 25 ubigeos y le pegamos las dos tablas con `how="left"`.
Cruzamos por `ubigeo` (no por nombre) y quitamos la columna `departamento` de las otras tablas para que no se repita.

In [54]:
tabla_final = (ubigeos
    .merge(decretos_ub.drop(columns="departamento"), on="ubigeo", how="left")
    .merge(lluvias_ub.drop(columns="departamento"), on="ubigeo", how="left"))
tabla_final.head()

,ubigeo,departamento,declaratorias,prorrogas,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,01,Amazonas,4,2,Chachapoyas,-6.23169,-77.86903,455.0,0
1,02,Áncash,5,5,Huaraz,-9.52614,-77.52869,1058.3,3
2,03,Apurímac,7,4,Abancay,-13.63426,-72.88380,670.6,2
3,04,Arequipa,7,6,Arequipa,-16.39899,-71.53747,235.3,1
4,05,Ayacucho,7,5,Ayacucho,-13.16380,-74.22345,467.5,1


### Paso 6 – Verificación: 25 filas, ceros donde no hubo declaratorias.

In [55]:
# Si algún departamento no tuviera decretos, quedaría vacío (NaN): le pone "0"
tabla_final[["declaratorias", "prorrogas"]] = (tabla_final[["declaratorias", "prorrogas"]].fillna(0).astype(int))
print("Filas:", len(tabla_final))                         # deben ser 25
print("Celdas vacías:", tabla_final.isna().sum().sum())    # deben ser 0
# Los departamentos sin declaratorias deben APARECER con 0, no desaparecer
tabla_final[tabla_final["declaratorias"] == 0][["ubigeo", "departamento", "declaratorias", "lluvia_total_mm"]]

Filas: 25
Celdas vacías: 0


,ubigeo,departamento,declaratorias,lluvia_total_mm
6,07,Callao,0,17.3
13,14,Lambayeque,0,25.6
19,20,Piura,0,139.4
23,24,Tumbes,0,497.0


Resultado 6: La tabla final (25 filas y sin celdas vacías) incluye con valor 0 a Callao, Lambayeque, Piura y Tumbes (sin declaratorias en 2024). 
Esto se logró utilizando how="left" sobre la lista oficial de ubigeos, evitando que dichos departamentos se omitieran, a diferencia de lo que habría ocurrido con un cruce how="inner".

### Paso 7 - Guardar `datos/tabla_final.csv`

In [56]:
tabla_final.to_csv("datos/tabla_final.csv", index=False, encoding="utf-8-sig")

# Verificación 1: miramos el archivo como texto plano (así lo verá cualquiera)
with open("datos/tabla_final.csv", encoding="utf-8-sig") as f:
    for _ in range(3):
        print(f.readline().strip())

# Verificación 2: al leerlo como texto, todos los ubigeos tienen 2 dígitos
control = pd.read_csv("datos/tabla_final.csv", dtype={"ubigeo": str})
print("\n¿Todos los ubigeos tienen 2 dígitos?:", control["ubigeo"].str.len().eq(2).all())

ubigeo,departamento,declaratorias,prorrogas,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
01,Amazonas,4,2,Chachapoyas,-6.23169,-77.86903,455.0,0
02,Áncash,5,5,Huaraz,-9.52614,-77.52869,1058.3,3

¿Todos los ubigeos tienen 2 dígitos?: True


### Paso 8 - Gráficos cpn Plotly
###8a. Lluvia total frente a declaratorias

In [59]:
import plotly.express as px

fig1 = px.scatter(
    tabla_final,
    x="lluvia_total_mm",
    y="declaratorias",
    text="departamento",
    hover_data=["capital", "dias_lluvia_fuerte", "prorrogas"],
    labels={"lluvia_total_mm": "Lluvia total en la capital (mm)",
            "declaratorias": "Número de declaratorias"},
    title="Lluvia total vs. declaratorias de emergencia por departamento (ene–may 2024)"
)
fig1.update_traces(textposition="top center")
fig1.show()

#### 8b. Departamentos con más declaratorias

Como Áncash, Junín, Moquegua y Puno empataron con 5 declaratorias cada uno (ocupando los puestos 8 al 11), el gráfico incluye a todos los empatados (11 departamentos) para no dejar fuera a nadie.

In [58]:
# Valor de corte: las declaratorias del departamento en el puesto 10
corte = tabla_final["declaratorias"].nlargest(10).min()

# Incluimos a todos los que alcanzan ese valor (así no se excluye a ningún empatado)
top10 = (tabla_final[tabla_final["declaratorias"] >= corte]
         .sort_values(["declaratorias", "departamento"], ascending=[False, True]))
print("Valor de corte:", corte, "| departamentos incluidos:", len(top10))

fig2 = px.bar(
    top10,
    x="departamento",
    y="declaratorias",
    text="declaratorias",
    labels={"departamento": "Departamento", "declaratorias": "Número de declaratorias"},
    title="Departamentos con más declaratorias por lluvias (ene–may 2024)")
fig2.show()


Valor de corte: 5 | departamentos incluidos: 11


## Paso 9 – Respuestas

### 9.1 ¿Los departamentos con más declaratorias son los que más llovió?

In [60]:
# Copia de la tabla para no modificar tabla_final
ranking = tabla_final.copy()

# Puestos: method="min" da el mismo puesto a los empatados
ranking["puesto_decl"] = ranking["declaratorias"].rank(ascending=False, method="min").astype(int)
ranking["puesto_lluvia"] = ranking["lluvia_total_mm"].rank(ascending=False, method="min").astype(int)

columnas = ["departamento", "declaratorias", "puesto_decl", "lluvia_total_mm", "puesto_lluvia"]

print("Departamentos con más declaratorias (puestos 1 a 3):")
print(ranking[ranking["puesto_decl"] <= 3][columnas].sort_values("puesto_lluvia").to_string(index=False))

print("\nLos 3 departamentos con más lluvia:")
print(ranking[ranking["puesto_lluvia"] <= 3][columnas].sort_values("puesto_lluvia").to_string(index=False))

# Correlación: 0 = sin relación, 1 = relación perfecta
print("\nCorrelación lluvia–declaratorias:",
      round(tabla_final["lluvia_total_mm"].corr(tabla_final["declaratorias"]), 2))

Departamentos con más declaratorias (puestos 1 a 3):
departamento  declaratorias  puesto_decl  lluvia_total_mm  puesto_lluvia
Huancavelica              7            1            819.2              7
    Apurímac              7            1            670.6             11
       Cusco              7            1            601.0             12
    Ayacucho              7            1            467.5             14
    Arequipa              7            1            235.3             17

Los 3 departamentos con más lluvia:
 departamento  declaratorias  puesto_decl  lluvia_total_mm  puesto_lluvia
       Loreto              3           16           1361.8              1
        Junín              5            8           1276.9              2
Madre de Dios              3           16           1149.5              3

Correlación lluvia–declaratorias: 0.2


Respuesta 9.1:
Más allá de encontrar un "top 3", los datos muestran un escenario particular: cinco departamentos empatan con 7 declaratorias cada uno de ellas (Apurímac, Arequipa, Ayacucho, Cusco y Huancavelica), todos pertenecientes a la sierra centro-sur.
Lo paradójico es que ninguno de ellos se encuentra entre los tres departamentos más lluviosos del país (Loreto, Junín y Madre de Dios):

•	Dentro del grupo de los cinco empatados, Huancavelica es el que registra más lluvia (puesto 7), seguido de Arequipa se ubica recién en el puesto 17.

•	A la inversa, Loreto (el departamento con mayor pluviosidad) ocupa el puesto 16 en cuanto a declaratorias.

En definitiva, la correlación estadística entre lluvia y declaratorias es de apenas 0.20 (una relación muy débil), lo que demuestra que llover más no implicó, por sí solo, recibir más declaratorias.

### 9.2 ¿Mucha lluvia y pocas declaratorias, o al revés?

Usamos la mediana de lluvia (497 mm) para separar "mucha" y "poca" lluvia.

In [61]:
mediana = tabla_final["lluvia_total_mm"].median()
print("Mediana de lluvia:", mediana, "mm")
columnas_92 = ["departamento", "lluvia_total_mm", "dias_lluvia_fuerte", "declaratorias", "prorrogas"]

# Mucha lluvia (sobre la mediana) y pocas declaratorias (3 o menos)
mucha_pocas = tabla_final[(tabla_final["lluvia_total_mm"] > mediana) & (tabla_final["declaratorias"] <= 3)]
print("\nMucha lluvia y pocas declaratorias:")
print(mucha_pocas[columnas_92].sort_values("lluvia_total_mm", ascending=False).to_string(index=False))

# Poca lluvia (bajo la mediana) y muchas declaratorias (5 o más)
poca_muchas = tabla_final[(tabla_final["lluvia_total_mm"] < mediana) & (tabla_final["declaratorias"] >= 5)]
print("\nPoca lluvia y muchas declaratorias:")
print(poca_muchas[columnas_92].sort_values("lluvia_total_mm").to_string(index=False))

Mediana de lluvia: 497.0 mm

Mucha lluvia y pocas declaratorias:
 departamento  lluvia_total_mm  dias_lluvia_fuerte  declaratorias  prorrogas
       Loreto           1361.8                  19              3          2
Madre de Dios           1149.5                  15              3          0
        Pasco            839.3                   1              3          3
   San Martín            681.7                   3              3          4
    Cajamarca            671.2                   0              3          3

Poca lluvia y muchas declaratorias:
departamento  lluvia_total_mm  dias_lluvia_fuerte  declaratorias  prorrogas
        Lima             12.2                   0              6          5
    Moquegua             29.1                   0              5          2
    Arequipa            235.3                   1              7          6
     Huánuco            416.7                   3              6          3
    Ayacucho            467.5                   1       

Respuesta 9.2:
Sí, al analizar los datos se aprecian contrastes muy marcados en ambas direcciones:
•	Mucha lluvia y pocas declaratorias: Departamentos con lluvias elevados y varios días de lluvia fuerte que, sin embargo, registraron apenas 3 declaratorias cada uno.
o	Loreto: 1,361.8 mm y 19 días de lluvia fuerte (2 prórrogas).
o	Madre de Dios: 1,149.5 mm y 15 días (0 prórrogas).
o	Pasco: 839.3 mm y 1 día (3 prórrogas).
•	Poca lluvia y muchas declaratorias: Regiones con registros de lluvia considerablemente menores que acumularon un número elevado de emergencias:
o	Lima: 12.2 mm y 0 días fuertes, pero con 6 declaratorias (y 5 prórrogas).
o	Moquegua: 29.1 mm y 0 días fuertes, con 5 declaratorias (y 2 prórrogas).
o	Arequipa: 235.3 mm y 1 día fuerte, con 7 declaratorias (y 6 prórrogas).
¿A qué se deben estas diferencias?
1.	El terreno y la vulnerabilidad: El impacto no depende solo de la castidad de lluvia, sino de dónde cae. En la sierra, una lluvia moderada puede activar quebradas y desatar graves huaicos y deslizamientos. En la selva, en cambio, las lluvias intensas son más frecuentes y en la mayoría de los casos la infraestructura está más adaptada.
2.	Gestión regional: Podría deberse a la capacidad operativa de cada gobierno regional. Las declaratorias no son automáticas: exigen que se soliciten formalmente y se evalúen los daños, por lo que influye mucho la iniciativa de cada administración.
3.	Historial de emergencias: El alto número de prórrogas en varios departamentos demuestra que muchas situaciones críticas no se originaron en el periodo analizado, sino que ya estaban declaradas con anterioridad y solo se extendieron.

### 9.3 ¿Peligro inminente o impacto de daños?

In [62]:
# Una fila por decreto de lluvias (Parte 1)
lluvias_decretos = pd.read_csv("datos/decretos_lluvias.csv")
print("Decretos de lluvias:", len(lluvias_decretos))   # control: 20

# Tabla cruzada tipo × motivo; margins=True agrega los totales
print(pd.crosstab(lluvias_decretos["tipo"], lluvias_decretos["motivo"], margins=True))

# a) Solo DECLARATORIAS (lo que pregunta el enunciado)
declaratorias = lluvias_decretos[lluvias_decretos["tipo"] == "declara"]
print("\n% de declaratorias por motivo:")
print((declaratorias["motivo"].value_counts(normalize=True) * 100).round(1))

# b) Todos los decretos (declaratorias + prórrogas), para comparar
print("\n% de todos los decretos por motivo:")
print((lluvias_decretos["motivo"].value_counts(normalize=True) * 100).round(1))

Decretos de lluvias: 20
motivo    impacto de daños  peligro inminente  All
tipo                                              
declara                 10                  0   10
prorroga                 8                  2   10
All                     18                  2   20

% de declaratorias por motivo:
motivo
impacto de daños    100.0
Name: proportion, dtype: float64

% de todos los decretos por motivo:
motivo
impacto de daños     90.0
peligro inminente    10.0
Name: proportion, dtype: float64


Respuesta 9.3:

Durante la temporada de lluvias de 2024, las cifras reflejan una respuesta estatal estrictamente reactiva:

Los 10 decretos de declaratoria de emergencia se emitieron exclusivamente por daños ya consumados (100%), sin registrar ninguno por peligro inminente.

Al sumar las extensiones (20 decretos en total), tenemos que el 90 % correspondió a daños y solo un 10 % (dos prórrogas) a peligro inminente. (Esto se corroboró al revisar manualmente una muestra de decretos en gob.pe).

¿Qué nos dice esto sobre la gestión del riesgo?
Que el Estado actúa tarde. A pesar de que la temporada de lluvias es un evento totalmente previsible año con año, las emergencias solo se oficializan cuando los daños ya están hechos. Si bien existe la figura del "peligro inminente" para adelantarse a los desastres y prevenir, en 2024 no se utilizó para lanzar ninguna alerta nueva.

### 9.4 Limitaciones a tener en cuenta en el análisis:

1.	La lluvia se midió en un solo punto: Los datos de precipitación se tomaron de las capitales (por ejemplo, Lima se midió en Huacho con apenas 12.2 mm), pero las emergencias reales ocurrieron en zonas específicas. Lo mismo sucede en Arequipa y Moquegua, donde las zonas altas reciben mucha más agua que las ciudades donde está la estación.
2.	Se cuentan decretos, no la magnitud del impacto: Analizamos cuántos decretos se emitieron, no a cuántos distritos o personas afectaron. 14 de los 20 decretos se menciona "varios distritos" sin detallar cuáles.
3.	Prórrogas y desfases de tiempo: La mitad de los decretos analizados (10 de 20) son prórrogas, y algunos todavía arrastraban emergencias de la temporada anterior (2023-2024). Además, un decreto suele publicarse semanas después de que llovió y causó el daño, un desfase temporal que los datos cruzados no alcanzan a medir.


### 9.5 Conclusión
Los datos de la temporada de lluvias 2024 nos muestra que el Estado no declaró emergencias necesariamente donde más llovió.

•	Las regiones con mayor lluvia como: Loreto y Madre de Dios, que superan los 1,100 mm, registraron apenas 3 declaratorias cada uno, mientras que los cinco departamentos con más emergencias como: Apurímac, Arequipa, Ayacucho, Cusco y Huancavelica, con 7 cada uno, no figuran entre los más lluviosos; asimismo, casos tan contrastantes como el de Lima, que con la menor cantidad de lluvia (12.2 mm) acumuló 6 declaratorias, lo que confirma que la relación estadística entre ambos factores es mínima (una correlación de apenas 0.20).

•	Se consideró más el daño que la cantidad de lluvia, las declaratorias responden sobre todo a la vulnerabilidad del territorio y a los daños reales sufridos, más que al volumen de lluvia acumulada.

•	Una reacción tardía: El 100 % de las declaratorias se emitieron por daños ya ocurridos, lo que evidencia una gestión del riesgo totalmente reactiva.
Es importante recordar que estos hallazgos tienen limitaciones, ya que la lluvia se midió principalmente en las capitales.